# 01.1 - Preprocessing v2 (segunda tentativa: 224×224)

Mesmo protocolo do `1.0` (NUAA oficial, mesmo split treino/validação/teste), mas com três mudanças. As imagens do `1.0` **não são alteradas**: esta versão grava em outra pasta (`data/processed_v2_224/`).

| | 1.0 (v1) | 1.1 (v2) |
|---|---|---|
| Tamanho | 160×160 | **224×224** |
| `margin` do MTCNN | 40 | **56** (mesmo enquadramento: a caixa do rosto cresce 33% nos dois) |
| Ajuste de brilho por imagem (min-max) | sim | **não** (`post_process=False`, pixels originais do recorte) |
| Formato de saída | `.jpg` (qualidade padrão, recomprime) | **`.png`** (sem perda) |
| Se o Colab cair | recomeça do zero | **retoma de onde parou** |

**O que é o `margin`:** quanto de área *em volta* do rosto detectado entra no recorte. No MTCNN ele é medido em pixels da imagem final, e a caixa do rosto cresce `margin / (tamanho - margin)` no total (metade para cada lado). Em 160 com `margin=40` e em 224 com `margin=56`, dá 33% nos dois casos.

Roda na CPU (não precisa de GPU).

In [ ]:
# Clona o repositório e instala as dependências.
# facenet-pytorch precisa de --no-deps porque o Colab não tem wheels
# pré-compiladas pras versões antigas de numpy/Pillow que ele pede.

!git clone https://github.com/laianemuckler/liveness-detection.git
%cd liveness-detection

!pip install -r requirements.txt --quiet
!pip install facenet-pytorch==2.6.0 --no-deps --quiet

In [ ]:
# Monta o Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Importa as funções do projeto

import os
import numpy as np
from PIL import Image
from tqdm import tqdm

from src.config import (
    TRAIN_V2_DIR, VAL_V2_DIR, TEST_V2_DIR,
    V2_IMAGE_SIZE, V2_MARGIN,
    LABEL_LIVE, LABEL_SPOOF,
)
from src.dataset import (
    load_protocol,
    split_train_validation,
    build_mtcnn,
    align_face,
)

print(f"Saída: {V2_IMAGE_SIZE}x{V2_IMAGE_SIZE}, margin={V2_MARGIN} "
      f"(caixa cresce {V2_MARGIN / (V2_IMAGE_SIZE - V2_MARGIN) * 100:.0f}%)")

In [ ]:
# Carrega o protocolo e monta o split (idêntico ao 1.0)

protocol = load_protocol()
train_entries, validation_entries = split_train_validation(protocol['train'])
test_entries = protocol['test']

print(f"Train:      {len(train_entries)} images")
print(f"Validation: {len(validation_entries)} images")
print(f"Test:       {len(test_entries)} images")

In [ ]:
# Cria o detector MTCNN v2.
# post_process=False: devolve os pixels do recorte (0-255) sem padronizar,
# então não há mais o ajuste de brilho/contraste por imagem do 1.0.

mtcnn = build_mtcnn(image_size=V2_IMAGE_SIZE, margin=V2_MARGIN, post_process=False)

In [ ]:
# Para cada imagem original:
#   - roda o MTCNN e recorta o rosto (imagem NOVA, a original não muda)
#   - salva em PNG (sem perda) em <saída>/<train|validation|test>/<live|spoof>/
#   - se o arquivo já existe, pula (retoma de onde parou se o Colab cair)
#   - se não detectar rosto, anota em _not_detected.txt e segue

LABEL_TO_FOLDER = {LABEL_LIVE: 'live', LABEL_SPOOF: 'spoof'}


def face_to_png_array(face_tensor):
    """Tensor (3,H,W) com pixels 0-255 -> array uint8 (H,W,3). Só arredonda; não reescala nada."""
    arr = face_tensor.permute(1, 2, 0).numpy()
    return np.clip(np.rint(arr), 0, 255).astype(np.uint8)


def process_split(entries, output_dir, split_name):
    for label in (LABEL_LIVE, LABEL_SPOOF):
        folder = os.path.join(output_dir, LABEL_TO_FOLDER[label])
        os.makedirs(folder, exist_ok=True)
        # remove temporários que sobraram de uma queda no meio de uma gravação
        for name in os.listdir(folder):
            if name.endswith('.tmp'):
                os.remove(os.path.join(folder, name))

    failures_path = os.path.join(output_dir, '_not_detected.txt')
    known_failures = set()
    if os.path.exists(failures_path):
        with open(failures_path) as f:
            known_failures = set(line.strip() for line in f if line.strip())

    saved = skipped = not_detected = 0
    for path, label, subject_id in tqdm(entries, desc=split_name):
        stem = os.path.splitext(os.path.basename(path))[0]
        out_path = os.path.join(output_dir, LABEL_TO_FOLDER[label], f"{subject_id}_{stem}.png")

        if os.path.exists(out_path):
            skipped += 1
            continue
        if path in known_failures:
            not_detected += 1
            continue

        face_tensor = align_face(mtcnn, path)
        if face_tensor is None:
            with open(failures_path, 'a') as f:
                f.write(path + '\n')
            not_detected += 1
            continue

        # grava num temporário e renomeia, para nunca ficar um PNG pela metade
        tmp_path = out_path + '.tmp'
        Image.fromarray(face_to_png_array(face_tensor)).save(tmp_path, format='PNG')
        os.replace(tmp_path, out_path)
        saved += 1

    print(f"{split_name}: {saved} novas | {skipped} já existiam | {not_detected} sem rosto detectado")

In [ ]:
# Processa os três conjuntos. Demora (~12 mil imagens). Se o Colab cair,
# rode as células de cima de novo e esta: ela continua de onde parou.

process_split(train_entries, TRAIN_V2_DIR, 'train')
process_split(validation_entries, VAL_V2_DIR, 'validation')
process_split(test_entries, TEST_V2_DIR, 'test')

In [ ]:
# Conferência: imagens salvas por pasta, contra o esperado do protocolo

for split_dir, split_name, entries in [
    (TRAIN_V2_DIR, 'train', train_entries),
    (VAL_V2_DIR, 'validation', validation_entries),
    (TEST_V2_DIR, 'test', test_entries),
]:
    n_live = len(os.listdir(os.path.join(split_dir, 'live')))
    n_spoof = len(os.listdir(os.path.join(split_dir, 'spoof')))
    print(f"{split_name}: live={n_live} | spoof={n_spoof} | total={n_live + n_spoof} (esperado {len(entries)})")

# Abre uma imagem para conferir tamanho e formato
sample_path = os.path.join(TRAIN_V2_DIR, 'live', sorted(os.listdir(os.path.join(TRAIN_V2_DIR, 'live')))[0])
img = Image.open(sample_path)
print(sample_path)
print("formato:", img.format, "| tamanho:", img.size, "| modo:", img.mode)

## Diagnóstico opcional: o 224 é ampliação?

Mede o tamanho das caixas do MTCNN numa amostra de 200 imagens. Se o recorte (caixa + margem) na imagem original for **menor que 224 px**, ir para 224 é ampliar (interpolação), e não ganho de detalhe. Não faz parte do pré-processamento; só responde a essa pergunta.

In [ ]:
import random

random.seed(0)
sample = random.sample(train_entries, 200)

sides = []
for path, label, subject_id in tqdm(sample, desc='caixas'):
    boxes, probs = mtcnn.detect(Image.open(path).convert('RGB'))
    if boxes is None:
        continue
    b = boxes[0]
    sides.append(((b[2] - b[0]) + (b[3] - b[1])) / 2)

sides = np.array(sides)
growth = 1 + V2_MARGIN / (V2_IMAGE_SIZE - V2_MARGIN)
crop_side = sides * growth

print(f"Lado médio da caixa do rosto: mediana {np.median(sides):.0f}px (mín {sides.min():.0f}, máx {sides.max():.0f})")
print(f"Lado do recorte com margem:   mediana {np.median(crop_side):.0f}px")
print(f"Imagens em que o recorte é menor que {V2_IMAGE_SIZE}px (ampliação): {np.mean(crop_side < V2_IMAGE_SIZE) * 100:.0f}%")